# Chapter 3 · 주택가격 ML 예측

**연구질문:** 주택가격 증가율 예측에서 트리 기반 ML은 선형모형과 기준선보다 나은가?

> 위에서 아래로 실행하면 끝까지 동작합니다. 온라인 실패 시 합성 대체(`DATA_KIND` 표시).

## 1. 핵심 개념·수식

**지도학습 형태의 예측:** $\hat y_{t}=f(y_{t-1},\dots,y_{t-L},\ \text{rolling}_{t-1},\ \text{calendar}_t,\ x_t)$. 모든 특성은 **예측 시점 $t$에 알 수 있는 정보**만 사용한다(롤링 통계는 반드시 한 칸 이동).

**시간순 검증:** 무작위 분할 대신 훈련 구간이 항상 검증 구간보다 앞서는 expanding-window 분할을 쓴다.

**기준선 대비 성능:** $\mathrm{MASE}<1$이면 1단계 나이브보다 낫다. 수익률처럼 예측력이 약한 자료는 **표본 외 $R^2_{OS}=1-\dfrac{\sum(y_t-\hat y_t)^2}{\sum(y_t-\bar y_{t-1})^2}$** ($\bar y_{t-1}$: 직전까지의 평균)로 평가한다.

**변수 중요도:** 순열 중요도는 특성을 섞었을 때 검증 오차가 늘어난 정도이며 인과적 중요도가 아니다. 상관된 특성이 있으면 해석이 불안정하다.

In [ ]:
import os, io, json, subprocess, sys, warnings, urllib.request, urllib.parse
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, optimize

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
SEED = 42
# 네트워크 없이 합성자료로만 실행하려면 환경변수 TSLAB_OFFLINE=1 로 실행하거나 아래를 False로 바꾼다.
USE_LIVE_DATA = os.environ.get("TSLAB_OFFLINE") != "1"
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True,
                     "axes.spines.top": False, "axes.spines.right": False})

def ensure(pkg, mod=None):
    # 없으면 설치를 시도한다. 실패해도 노트북은 계속 실행되고 해당 선택 기능만 건너뛴다.
    try:
        __import__(mod or pkg); return True
    except Exception:
        if not USE_LIVE_DATA:
            return False
        try:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True, timeout=300)
            __import__(mod or pkg); return True
        except Exception as e:
            print(f"[건너뜀] {pkg} 설치/가져오기 실패: {type(e).__name__}")
            return False

def fetch_bytes(url, timeout=60):
    req = urllib.request.Request(url, headers={"User-Agent": "ts-lab-course/1.0 (education)"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

# ---- 평가 지표와 기준선 ----
def mae(a, f):  return float(np.mean(np.abs(np.asarray(a, float) - np.asarray(f, float))))
def rmse(a, f): return float(np.sqrt(np.mean((np.asarray(a, float) - np.asarray(f, float)) ** 2)))
def mase(a, f, train, m=1):
    train = np.asarray(train, float)
    scale = np.mean(np.abs(train[m:] - train[:-m]))
    return mae(a, f) / scale
def naive_fc(train, h):  return np.repeat(np.asarray(train, float)[-1], h)
def snaive_fc(train, h, m):
    train = np.asarray(train, float)
    return np.tile(train[-m:], int(np.ceil(h / m)))[:h]
def drift_fc(train, h):
    train = np.asarray(train, float)
    slope = (train[-1] - train[0]) / (len(train) - 1)
    return train[-1] + slope * np.arange(1, h + 1)
def show(df, digits=4):
    print(df.round(digits).to_string())

In [ ]:
def parse_fhfa_master(raw):
    # FHFA hpi_master.csv 파싱. 스키마(열 이름)는 버전에 따라 달라질 수 있어 방어적으로 처리하고, 실패하면 예외를 낸다.
    df = pd.read_csv(io.BytesIO(raw) if isinstance(raw, (bytes, bytearray)) else io.StringIO(raw))
    df.columns = [c.strip().lower() for c in df.columns]
    if not {"yr", "period"}.issubset(df.columns):
        raise ValueError("열(yr, period) 없음: " + ", ".join(df.columns[:12]))
    col = "index_nsa" if "index_nsa" in df.columns else None
    if col is None:
        raise ValueError("index_nsa 열 없음")
    if "frequency" in df.columns:
        df = df[df["frequency"].astype(str).str.lower().str.startswith("month")]
    keys = [k for k in ("hpi_type", "hpi_flavor", "level", "place_id", "place_name") if k in df.columns]
    g = df.groupby(keys).size().sort_values(ascending=False)
    if keys:
        df = df.merge(g.index[:1].to_frame(index=False), on=keys)   # 관측이 가장 많은 한 계열
    s = pd.Series(pd.to_numeric(df[col], errors="coerce").values,
                  index=pd.to_datetime(dict(year=df["yr"].astype(int), month=df["period"].astype(int), day=1)))
    s = s.dropna().sort_index()
    if len(s) < 100:
        raise ValueError("관측 수 부족")
    return s

def synthetic_house(n=300, seed=SEED):
    rng = np.random.default_rng(seed); t = np.arange(n)
    cycle = 6 * np.sin(2 * np.pi * t / 120)
    seas = 1.2 * np.sin(2 * np.pi * (t % 12) / 12 - 1.0)
    shock = np.cumsum(rng.normal(0, 0.35, n))
    return pd.Series(100 + 0.35 * t + cycle + seas + shock, index=pd.date_range("2000-01-01", periods=n, freq="MS"))

def synthetic_volume(n=240, seed=SEED + 1):
    rng = np.random.default_rng(seed); t = np.arange(n)
    seas = 18 * np.sin(2 * np.pi * (t % 12) / 12 - 1.5)
    return pd.Series(np.maximum(200 + 0.2 * t + seas + np.cumsum(rng.normal(0, 1.2, n)) + rng.normal(0, 6, n), 10),
                     index=pd.date_range("2005-01-01", periods=n, freq="MS"))

def load_realestate(which="price"):
    if which == "volume":
        return dict(y=synthetic_volume().rename("volume"), m=12, freq="MS", label="월별 주택거래량(합성)", unit="건",
                    kind="SYNTHETIC (NOT real data): 검증된 무료 거래량 출처 없음")
    s, kind = None, "SYNTHETIC (NOT real data): 추세+주기+계절+확률적 충격 주택가격지수"
    if USE_LIVE_DATA:
        try:
            s = parse_fhfa_master(fetch_bytes("https://www.fhfa.gov/hpi/download/monthly/hpi_master.csv", timeout=90))
            kind = "REAL: FHFA HPI (index_nsa, 계열 선택 규칙은 parse_fhfa_master 참고)"
        except Exception as e:
            print("FHFA 수집/파싱 실패 -> 합성 대체:", type(e).__name__, str(e)[:120])
    if s is None:
        s = synthetic_house()
    return dict(y=s.rename("hpi"), m=12, freq="MS", label="주택가격지수", unit="지수", kind=kind)

D = load_realestate("price")
y, DATA_KIND, M, FREQ = D["y"], D["kind"], D["m"], D["freq"]
print("DATA_KIND:", DATA_KIND); print("관측 수:", len(y), "| 기간:", y.index.min().date(), "~", y.index.max().date())

## 2. 특성 구성·모형·평가

In [ ]:
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import (RandomForestRegressor, HistGradientBoostingRegressor,
                              RandomForestClassifier, HistGradientBoostingClassifier)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve

def calendar_features(idx, freq):
    out = pd.DataFrame(index=idx)
    if freq in ("D", "B", "h"):
        out["dow"] = idx.dayofweek
    if freq == "h":
        out["hour"] = idx.hour
    out["month_sin"] = np.sin(2 * np.pi * idx.month / 12); out["month_cos"] = np.cos(2 * np.pi * idx.month / 12)
    return out

def lag_features(s, lags, rolls=()):
    X = pd.DataFrame({f"lag_{k}": s.shift(k) for k in lags})
    for w in rolls:
        X[f"rmean_{w}"] = s.shift(1).rolling(w).mean()          # shift(1): 현재 값을 쓰지 않는다(누수 방지)
        X[f"rstd_{w}"] = s.shift(1).rolling(w).std()
    return X

def get_models():
    models = {
        "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
        "RandomForest": RandomForestRegressor(n_estimators=150, min_samples_leaf=5, max_features=0.6, n_jobs=-1, random_state=SEED),
        "HistGB": HistGradientBoostingRegressor(max_iter=250, learning_rate=0.05, random_state=SEED),
    }
    if ensure("xgboost"):                                         # 선택: 설치돼 있으면 XGBoost도 비교
        from xgboost import XGBRegressor
        models["XGBoost"] = XGBRegressor(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8,
                                         random_state=SEED, n_jobs=-1, verbosity=0)
    return models

def fit_eval(X, target, n_test, models=None):
    # 앞쪽으로 학습, 마지막 n_test 관측으로 평가 (1단계 예측, 시차 입력은 실제값)
    data = pd.concat([X, target.rename("target")], axis=1).dropna()
    tr, te = data.iloc[:-n_test], data.iloc[-n_test:]
    Xtr, ytr, Xte, yte = tr.drop(columns="target"), tr["target"], te.drop(columns="target"), te["target"]
    models = models or get_models(); preds = {}
    for name, mdl in models.items():
        mdl.fit(Xtr, ytr); preds[name] = pd.Series(mdl.predict(Xte), index=te.index)
    return dict(Xtr=Xtr, ytr=ytr, Xte=Xte, yte=yte, preds=preds, models=models, data=data)

def cv_rmse(X, target, models, n_splits=4, test_size=None):
    data = pd.concat([X, target.rename("target")], axis=1).dropna()
    test_size = test_size or max(30, len(data) // 10)
    tss = TimeSeriesSplit(n_splits=n_splits, test_size=test_size); out = {k: [] for k in models}
    from sklearn.base import clone
    for tr_i, te_i in tss.split(data):
        a, b = data.iloc[tr_i], data.iloc[te_i]
        for name, mdl in models.items():
            m_ = clone(mdl).fit(a.drop(columns="target"), a["target"])
            out[name].append(rmse(b["target"], m_.predict(b.drop(columns="target"))))
    return pd.DataFrame(out).agg(["mean", "std"]).T.rename(columns={"mean": "CV_RMSE_mean", "std": "CV_RMSE_sd"})

In [ ]:
# 결과표: 기준선과 모델, 마지막 구간 평가 + 시간순 CV
def report(res, baselines, train_for_scale):
    rows = {}
    for name, f in {**baselines, **res["preds"]}.items():
        rows[name] = {"MAE": mae(res["yte"], f), "RMSE": rmse(res["yte"], f),
                      "MASE": mase(res["yte"], f, train_for_scale, m=1)}
    tab = pd.DataFrame(rows).T.sort_values("RMSE"); show(tab); return tab

In [ ]:
g = np.log(y).diff() * 100                                           # 월 성장률(%)
cal = calendar_features(g.index, "MS")[["month_sin", "month_cos"]]
X = pd.concat([lag_features(g, list(range(1, 13)), rolls=(3, 12)), cal], axis=1)
n_test = 48
res = fit_eval(X, g, n_test)
yte = res["yte"]
base = {"naive(lag1)": g.shift(1).loc[yte.index], "seasonal naive(lag12)": g.shift(12).loc[yte.index]}
tab = report(res, base, res["ytr"].to_numpy())
print("\n시간순 CV (RMSE):"); show(cv_rmse(X, g, get_models(), n_splits=4, test_size=36))

**읽는 법:** 가격 **수준**을 예측하면 지수가 느리게 움직여 나이브도 매우 정확해 보인다. 그래서 여기서는 **증가율**을 예측 대상으로 삼았다. 지수는 평활·개정 절차를 거친 자료일 수 있어 자기상관이 인위적으로 클 수 있다(확인 필요).

## 3. 결과 해석 작성
1. 기준선(나이브·계절 나이브·역사평균)보다 나은 모델이 있는가? 개선폭은 CV 변동성($sd$) 대비 큰가?
2. 변수 중요도가 도메인 지식과 일관되는가? (인과적 해석 금지)
3. 가장 위험한 누수 경로 3개를 적고, 이 노트북에서 어떻게 막았는지(롤링 shift, 학습구간 통계 스케일, 시간순 분할)를 서술한다.
4. `DATA_KIND`가 합성이면 결과는 **방법 점검**이다. 실제 자료로 바꿨을 때 달라질 점을 적는다.

**제출:** 결과표(기준선 포함), CV 표, 중요도 그래프/표, 해석 단락, 한계.